# 01. Data Cleaning and Quality Assessment

This notebook validates the raw IBM Telco Customer Churn dataset, documents quality issues, converts `TotalCharges` to numeric, and writes a clean dataset and a machine-readable quality report.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from src.data_cleaning import load_raw_data, clean_data, save_clean_data, \
    save_quality_report, write_data_quality_report
from src import DATA_PROCESSED_DIR

In [2]:
df = load_raw_data()
print(f'Raw dataset shape: {df.shape}')
df.head(5)

Raw dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

## Initial quality observations

- `TotalCharges` is read as text; it must be converted to numeric.
- `SeniorCitizen` uses 0/1 codes; these will be relabelled for readability.
- The rest of the columns look structurally consistent.

In [4]:
# Detect blank / whitespace-only entries (they are not NaN to pandas)
blanks = df.apply(lambda s: s.astype('string').str.strip().eq('')).sum()
print('Columns with blank entries:')
print(blanks[blanks > 0])

Columns with blank entries:
TotalCharges    11
dtype: Int64


In [5]:
# Run the full cleaning routine
clean, report = clean_data(df)
print(f'Cleaned shape: {clean.shape}')
print()
print('Blank TotalCharges filled with 0:', report['blank_total_charges_filled_with_zero'])
print('Duplicated customer IDs:', report['customer_id_check']['duplicated_customer_ids'])
print('Missing customer IDs:', report['customer_id_check']['missing_customer_ids'])

Cleaned shape: (7043, 21)

Blank TotalCharges filled with 0: 11
Duplicated customer IDs: 0
Missing customer IDs: 0


In [6]:
pd.DataFrame(report['outlier_investigation']).T

,count,min,max,mean,median,iqr_lower_fence,iqr_upper_fence,iqr_outlier_count,zscore_outlier_count_3sd
tenure,7043.0,0.00,72.00,32.371149,29.00,-60.000,124.000,0.0,0.0
MonthlyCharges,7043.0,18.25,118.75,64.761692,70.35,-46.025,171.375,0.0,0.0
TotalCharges,7043.0,0.00,8684.80,2279.734304,1394.55,-4683.525,8868.675,0.0,0.0


In [7]:
# Confirm TotalCharges is now numeric
print(clean['TotalCharges'].dtype)
clean[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)

Float64


,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.0
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.0
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.6
max,72.00,118.75,8684.8


In [8]:
out = save_clean_data(clean)
save_quality_report(report)
write_data_quality_report(report)
print('Saved:', out)

Saved: /workspace/customer-churn-analytics/data/processed/customer_churn_clean.csv


## Interpretation

- The dataset is complete at 7,043 rows; the only quality issue is 11 whitespace-only `TotalCharges` values, all belonging to tenure-0 customers, which we filled with 0 after verifying tenure.
- No duplicate `customerID` values exist.
- Outliers exist on charges and tenure but are deliberately kept: this is an observational study of the entire customer base.